Extract Thermo files and save data to parquet files.
Plot ozone data.

joerg.klausen@meteoswiss.ch

In [ ]:
import os
import polars as pl

from processing.thermo import Thermo

log = "/home/zue/users/jkl/Public/git/gawkenya/logging/thermo.log"
thermo = Thermo(log=log)

In [ ]:
# Extract and collate Thermo files
instr = "tei49c"
source = f"/product_data/data/pay/Kenya/MKN/incoming/{instr}"
target = "/home/zue/users/jkl/Public/git/gawkenya/data/level1"
# archive = f"/product_data/data/pay/Kenya/MKN/archive/{instr}"
archive = None
issues = f"/product_data/data/pay/Kenya/MKN/incoming_with_issues/{instr}"

# for base in ["2021", "2022", "2023"]:
for base in ["2022"]:
    if os.path.exists(source):
        thermo.compile_thermo_to_parquet(source=source, target=target, base=base, archive=archive, issues=issues)

In [ ]:
# Unarchive files
from housekeeping import organize_files
instr = "tei49c"
source = f"/product_data/data/pay/Kenya/MKN/incoming/{instr}"
archive = f"/product_data/data/pay/Kenya/MKN/archive/{instr}"
n = organize_files.move_files(source=os.path.join(archive, "2021"), target=source, pattern="tei49c-\d{12}.(zip|dat)")
n = organize_files.move_files(source=os.path.join(archive, "2022"), target=source, pattern="tei49c-\d{12}.(zip|dat)")
n = organize_files.move_files(source=os.path.join(archive, "2023"), target=source, pattern="tei49c-\d{12}.(zip|dat)")

In [ ]:
instr = "tei49i"
source = f"/product_data/data/pay/Kenya/MKN/incoming/{instr}"
target = "/home/zue/users/jkl/Public/git/gawkenya/results/level_1_data"
archive = f"/product_data/data/pay/Kenya/MKN/archive/{instr}"
issues = f"/product_data/data/pay/Kenya/MKN/incoming_with_issues/{instr}"

for base in ["2021", "2022", "2023"]:
    if os.path.exists(source):
        thermo.compile_thermo_to_parquet(source=source, target=target, base=base, archive=archive, issues=issues)

In [ ]:
instr = "tei49i_2"
source = f"/product_data/data/pay/Kenya/MKN/incoming/{instr}"
target = "/home/zue/users/jkl/Public/git/gawkenya/results/level_1_data"
archive = f"/product_data/data/pay/Kenya/MKN/archive/{instr}"
issues = f"/product_data/data/pay/Kenya/MKN/incoming_with_issues/{instr}"

for base in ["2022", "2023"]:
    if os.path.exists(source):
        thermo.compile_thermo_to_parquet(source=source, target=target, base=base, archive=archive, issues=issues)

In [ ]:
# quick'n'dirty plot of one variable
import matplotlib as plt

df = pl.read_parquet("/home/zue/users/jkl/Public/git/gawkenya/results/level_1_data/2021/tei49c.parquet")
display(df.describe())

variable="o3"
df_clean, errors = thermo.remove_extremes(df, variable=variable, q=0.0001)
thermo.plot_data(df_clean, variable=variable)

In [ ]:
# # Concatenate the individual DataFrames into a single one
# combined_data = pl.concat(dataframes)

# # Assuming you have datetime columns, replace 'datetime_column' with the actual column name.
# datetime_column = 'timestamp'

# # Perform aggregation by datetime_column
# agg_data = (
#     combined_data
#     .with_column(combined_data[datetime_column].cast(pl.Date32))
#     .groupby(datetime_column)
#     .agg(pl.sum(combined_data['value_column']))
#     .sort(datetime_column)
# )